In [1]:
import json
import os
import csv
import random
import numpy as np

try:
    import tensorflow as tf
    import editdistance
except ImportError:
    print("ERROR: This script requires tensorflow and editdistance.")
    print("Install with: pip install tensorflow editdistance")
    exit(1)

from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'

BASELINE_DATASET_PATH  = f'{DRIVE}/multilingual_g2p_dataset.txt'
CLUSTERED_DATASET_PATH = f'{DRIVE}/multilingual_g2p_clustered.txt'
BASELINE_WEIGHTS_PATH  = f'{DRIVE}/best_g2p_transformer.weights.h5'
CLUSTERED_WEIGHTS_PATH = f'{DRIVE}/g2p_clustered_model.weights.h5'
SRC_TOKENIZER_PATH     = f'{DRIVE}/src_tokenizer.json'
TGT_TOKENIZER_PATH     = f'{DRIVE}/tgt_tokenizer.json'
TGT_TOKENIZER_CLUST    = f'{DRIVE}/tgt_tokenizer_clustered.json'
REPORT_PATH            = f'{DRIVE}/results/evaluation_report.md'
CSV_PATH               = f'{DRIVE}/results/comparison_table.csv'


class Tokenizer:
    def __init__(self):
        self.pad_token = '<pad>'
        self.unk_token = '<unk>'
        self.sos_token = '<sos>'
        self.eos_token = '<eos>'
        self.s2i = {self.pad_token: 0, self.unk_token: 1, self.sos_token: 2, self.eos_token: 3}
        self.i2s = {0: self.pad_token, 1: self.unk_token, 2: self.sos_token, 3: self.eos_token}
        self.vocab_size = 4

    def fit(self, sentences):
        for sentence in sentences:
            for token in sentence:
                if token not in self.s2i:
                    self.s2i[token] = self.vocab_size
                    self.i2s[self.vocab_size] = token
                    self.vocab_size += 1

    def encode(self, sentence, add_special=True):
        encoded = [self.s2i.get(tok, self.s2i[self.unk_token]) for tok in sentence]
        if add_special:
            encoded = [self.s2i[self.sos_token]] + encoded + [self.s2i[self.eos_token]]
        return encoded

    def decode(self, indices, remove_special=True):
        special = {self.pad_token, self.sos_token, self.eos_token}
        result = []
        for idx in indices:
            tok = self.i2s.get(int(idx), self.unk_token)
            if remove_special and tok in special:
                continue
            result.append(tok)
        return result

    def save(self, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'s2i': self.s2i, 'i2s': {int(k): v for k, v in self.i2s.items()}, 'vocab_size': self.vocab_size}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        tok.s2i = data['s2i']
        tok.i2s = {int(k): v for k, v in data['i2s'].items()}
        tok.vocab_size = data['vocab_size']
        return tok


def get_positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, :, :], dtype=tf.float32)

def create_padding_mask(seq):
    mask = tf.cast(tf.math.equal(seq, 0), tf.float32)
    return mask[:, tf.newaxis, tf.newaxis, :]

def create_look_ahead_mask(size):
    mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    return mask

def create_masks(src, tgt):
    enc_padding_mask = create_padding_mask(src)
    dec_padding_mask = create_padding_mask(src)
    look_ahead_mask = create_look_ahead_mask(tf.shape(tgt)[1])
    dec_target_padding_mask = create_padding_mask(tgt)
    combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)
    return enc_padding_mask, combined_mask, dec_padding_mask

def scaled_dot_product_attention(q, k, v, mask):
    matmul_qk = tf.matmul(q, k, transpose_b=True)
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled = matmul_qk / tf.math.sqrt(dk)
    if mask is not None:
        scaled += (mask * -1e9)
    weights = tf.nn.softmax(scaled, axis=-1)
    return tf.matmul(weights, v)

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs, **kwargs):
        v, k, q, mask = inputs['v'], inputs['k'], inputs['q'], inputs['mask']
        batch_size = tf.shape(q)[0]
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)
        output = scaled_dot_product_attention(q, k, v, mask)
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        concat = tf.reshape(output, (batch_size, -1, self.d_model))
        return self.dense(concat)

def point_wise_ffn(d_model, dff):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(dff, activation='relu'),
        tf.keras.layers.Dense(d_model)
    ])

class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_ffn(d_model, dff)
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, **kwargs):
        x, mask = inputs
        attn = self.mha({'v': x, 'k': x, 'q': x, 'mask': mask})
        attn = self.dropout1(attn, training=training)
        out1 = self.ln1(x + attn)
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out, training=training)
        return self.ln2(out1 + ffn_out)

class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_ffn(d_model, dff)
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout3 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, **kwargs):
        x, enc_output, look_ahead_mask, padding_mask = inputs
        attn1 = self.mha1({'v': x, 'k': x, 'q': x, 'mask': look_ahead_mask})
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.ln1(attn1 + x)
        attn2 = self.mha2({'v': enc_output, 'k': enc_output, 'q': out1, 'mask': padding_mask})
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.ln2(attn2 + out1)
        ffn_out = self.ffn(out2)
        ffn_out = self.dropout3(ffn_out, training=training)
        return self.ln3(ffn_out + out2)

class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size, max_pos_enc, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(input_vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(max_pos_enc, d_model)
        self.enc_layers = [EncoderLayer(d_model, num_heads, dff, dropout_rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, **kwargs):
        x, mask = inputs
        seq_len = tf.shape(x)[1]
        x = self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        for layer in self.enc_layers:
            x = layer((x, mask), training=training)
        return x

class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size, max_pos_enc, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(target_vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(max_pos_enc, d_model)
        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, dropout_rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, **kwargs):
        x, enc_output, look_ahead_mask, padding_mask = inputs
        seq_len = tf.shape(x)[1]
        x = self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        for layer in self.dec_layers:
            x = layer((x, enc_output, look_ahead_mask, padding_mask), training=training)
        return x

class Transformer(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff,
                 input_vocab_size, target_vocab_size,
                 pe_input, pe_target, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.encoder = Encoder(num_layers, d_model, num_heads, dff,
                               input_vocab_size, pe_input, dropout_rate)
        self.decoder = Decoder(num_layers, d_model, num_heads, dff,
                               target_vocab_size, pe_target, dropout_rate)
        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, inputs, training=False, **kwargs):
        src, tgt = inputs
        enc_padding_mask, look_ahead_mask, dec_padding_mask = create_masks(src, tgt)
        enc_output = self.encoder((src, enc_padding_mask), training=training)
        dec_output = self.decoder((tgt, enc_output, look_ahead_mask, dec_padding_mask), training=training)
        return self.final_layer(dec_output)


def load_dataset(file_path):
    """Load G2P dataset. Returns (src_sentences, tgt_sentences)."""
    src_sentences, tgt_sentences = [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            src_str = parts[0].strip()
            tgt_str = parts[1].strip()
            if src_str.startswith('<') and '>' in src_str:
                end_idx = src_str.index('>') + 1
                lang_tag = src_str[:end_idx]
                word = src_str[end_idx:].strip()
            else:
                lang_tag = '<UNK>'
                word = src_str
            src_seq = [lang_tag] + list(word)
            tgt_seq = tgt_str.split()
            src_sentences.append(src_seq)
            tgt_sentences.append(tgt_seq)
    return src_sentences, tgt_sentences


def greedy_decode(model, src, tgt_tokenizer, max_len=100):
    """Greedy autoregressive decoding."""
    sos_id = tgt_tokenizer.s2i[tgt_tokenizer.sos_token]
    eos_id = tgt_tokenizer.s2i[tgt_tokenizer.eos_token]
    decoder_input = tf.expand_dims([sos_id], 0)

    for _ in range(max_len):
        predictions = model((src, decoder_input), training=False)
        next_token = tf.argmax(predictions[:, -1, :], axis=-1)
        next_token = tf.cast(tf.expand_dims(next_token, 0), tf.int32)
        decoder_input = tf.concat([decoder_input, next_token], axis=-1)
        if next_token.numpy()[0][0] == eos_id:
            break

    return decoder_input.numpy()[0].tolist()


def evaluate_model(model, test_src_enc, test_tgt_enc, tgt_tokenizer, model_name, num_samples=None):
    """Evaluate model on test set. Returns dict with PER, WER, sample predictions."""
    if num_samples is None:
        num_samples = len(test_src_enc)

    total_per = 0.0
    total_wer = 0.0
    count = 0
    samples = []

    for i in range(min(num_samples, len(test_src_enc))):
        src_input = tf.expand_dims(test_src_enc[i], 0)
        predicted_ids = greedy_decode(model, src_input, tgt_tokenizer, max_len=50)

        pred_tokens = tgt_tokenizer.decode(predicted_ids, remove_special=True)
        true_tokens = tgt_tokenizer.decode(test_tgt_enc[i].tolist(), remove_special=True)

        per = editdistance.eval(pred_tokens, true_tokens) / max(len(true_tokens), 1)
        total_per += per

        wer = 0.0 if pred_tokens == true_tokens else 1.0
        total_wer += wer
        count += 1

        if i < 10:
            samples.append({
                "predicted": " ".join(pred_tokens),
                "expected": " ".join(true_tokens),
                "per": per,
            })

        if (i + 1) % 500 == 0:
            print(f"  [{model_name}] Evaluated {i+1}/{min(num_samples, len(test_src_enc))} samples...")

    avg_per = total_per / count
    avg_wer = total_wer / count

    return {
        "model_name": model_name,
        "per": avg_per,
        "wer": avg_wer,
        "num_samples": count,
        "samples": samples,
    }


def count_parameters(model):
    """Count total trainable parameters."""
    return sum(np.prod(v.shape) for v in model.trainable_variables)


def get_file_size_mb(path):
    """Get file size in MB."""
    if os.path.exists(path):
        return os.path.getsize(path) / (1024 * 1024)
    return 0.0


def main():
    print("=" * 60)
    print("TASK 3: BASELINE vs. CLUSTERED G2P EVALUATION")
    print("=" * 60)

    print("\nLoading baseline dataset...")
    baseline_src, baseline_tgt = load_dataset(BASELINE_DATASET_PATH)
    print(f"  Loaded {len(baseline_src)} baseline samples")

    print("Loading clustered dataset...")
    clustered_src, clustered_tgt = load_dataset(CLUSTERED_DATASET_PATH)
    print(f"  Loaded {len(clustered_src)} clustered samples")

    random.seed(42)
    indices = list(range(len(baseline_src)))
    random.shuffle(indices)

    train_size = int(len(indices) * 0.8)
    val_size = int(len(indices) * 0.1)

    train_idx = indices[:train_size]
    test_idx = indices[train_size + val_size:]

    train_src_b = [baseline_src[i] for i in train_idx]
    train_tgt_b = [baseline_tgt[i] for i in train_idx]
    test_src_b  = [baseline_src[i] for i in test_idx]
    test_tgt_b  = [baseline_tgt[i] for i in test_idx]

    train_src_c = [clustered_src[i] for i in train_idx]
    train_tgt_c = [clustered_tgt[i] for i in train_idx]
    test_src_c  = [clustered_src[i] for i in test_idx]
    test_tgt_c  = [clustered_tgt[i] for i in test_idx]

    print(f"  Test set size: {len(test_idx)} samples")

    src_tokenizer = Tokenizer()
    src_tokenizer.fit(train_src_b)
    tgt_tokenizer_b = Tokenizer()
    tgt_tokenizer_b.fit(train_tgt_b)

    tgt_tokenizer_c = Tokenizer()
    tgt_tokenizer_c.fit(train_tgt_c)

    print(f"\n  Baseline  — src vocab: {src_tokenizer.vocab_size}, tgt vocab: {tgt_tokenizer_b.vocab_size}")
    print(f"  Clustered — src vocab: {src_tokenizer.vocab_size}, tgt vocab: {tgt_tokenizer_c.vocab_size}")

    def encode_and_pad(src_sents, tgt_sents, src_tok, tgt_tok):
        src_encoded = [src_tok.encode(s) for s in src_sents]
        tgt_encoded = [tgt_tok.encode(t) for t in tgt_sents]
        src_padded = tf.keras.preprocessing.sequence.pad_sequences(src_encoded, padding='post', value=0)
        tgt_padded = tf.keras.preprocessing.sequence.pad_sequences(tgt_encoded, padding='post', value=0)
        return src_padded, tgt_padded

    train_src_enc_b, train_tgt_enc_b = encode_and_pad(train_src_b, train_tgt_b, src_tokenizer, tgt_tokenizer_b)
    test_src_enc_b, test_tgt_enc_b   = encode_and_pad(test_src_b, test_tgt_b, src_tokenizer, tgt_tokenizer_b)

    train_src_enc_c, train_tgt_enc_c = encode_and_pad(train_src_c, train_tgt_c, src_tokenizer, tgt_tokenizer_c)
    test_src_enc_c, test_tgt_enc_c   = encode_and_pad(test_src_c, test_tgt_c, src_tokenizer, tgt_tokenizer_c)

    NUM_LAYERS = 3
    D_MODEL = 128
    NUM_HEADS = 4
    DFF = 512
    DROPOUT_RATE = 0.1

    print("\nBuilding baseline model...")
    MAX_SRC_LEN_B = train_src_enc_b.shape[1] + 50
    MAX_TGT_LEN_B = train_tgt_enc_b.shape[1] + 50

    baseline_model = Transformer(
        num_layers=NUM_LAYERS, d_model=D_MODEL, num_heads=NUM_HEADS, dff=DFF,
        input_vocab_size=src_tokenizer.vocab_size,
        target_vocab_size=tgt_tokenizer_b.vocab_size,
        pe_input=MAX_SRC_LEN_B, pe_target=MAX_TGT_LEN_B,
        dropout_rate=DROPOUT_RATE
    )

    dummy_src = tf.zeros((1, 10), dtype=tf.int32)
    dummy_tgt = tf.zeros((1, 10), dtype=tf.int32)
    _ = baseline_model((dummy_src, dummy_tgt), training=False)

    print(f"  Loading weights from: {BASELINE_WEIGHTS_PATH}")
    baseline_model.load_weights(BASELINE_WEIGHTS_PATH)
    baseline_params = count_parameters(baseline_model)
    baseline_size_mb = get_file_size_mb(BASELINE_WEIGHTS_PATH)
    print(f"  Parameters: {baseline_params:,}")
    print(f"  Checkpoint: {baseline_size_mb:.2f} MB")

    print("\nBuilding clustered model...")
    MAX_SRC_LEN_C = train_src_enc_c.shape[1] + 50
    MAX_TGT_LEN_C = train_tgt_enc_c.shape[1] + 50

    clustered_model = Transformer(
        num_layers=NUM_LAYERS, d_model=D_MODEL, num_heads=NUM_HEADS, dff=DFF,
        input_vocab_size=src_tokenizer.vocab_size,
        target_vocab_size=tgt_tokenizer_c.vocab_size,
        pe_input=MAX_SRC_LEN_C, pe_target=MAX_TGT_LEN_C,
        dropout_rate=DROPOUT_RATE
    )

    _ = clustered_model((dummy_src, dummy_tgt), training=False)

    print(f"  Loading weights from: {CLUSTERED_WEIGHTS_PATH}")
    clustered_model.load_weights(CLUSTERED_WEIGHTS_PATH)
    clustered_params = count_parameters(clustered_model)
    clustered_size_mb = get_file_size_mb(CLUSTERED_WEIGHTS_PATH)
    print(f"  Parameters: {clustered_params:,}")
    print(f"  Checkpoint: {clustered_size_mb:.2f} MB")

    print("\n" + "=" * 60)
    print("EVALUATING BASELINE MODEL...")
    print("=" * 60)
    baseline_results = evaluate_model(
        baseline_model, test_src_enc_b, test_tgt_enc_b,
        tgt_tokenizer_b, "Baseline"
    )

    print("\n" + "=" * 60)
    print("EVALUATING CLUSTERED MODEL...")
    print("=" * 60)
    clustered_results = evaluate_model(
        clustered_model, test_src_enc_c, test_tgt_enc_c,
        tgt_tokenizer_c, "Clustered"
    )

    original_phoneme_count = tgt_tokenizer_b.vocab_size - 4  # minus special tokens
    cluster_count = tgt_tokenizer_c.vocab_size - 4
    vocab_reduction = (1 - cluster_count / original_phoneme_count) * 100

    print("\n" + "=" * 60)
    print("COMPARISON RESULTS")
    print("=" * 60)

    print(f"\n{'Metric':<30} {'Baseline':>12} {'Clustered':>12} {'Δ':>12}")
    print("-" * 70)

    per_delta = clustered_results['per'] - baseline_results['per']
    wer_delta = clustered_results['wer'] - baseline_results['wer']
    param_delta = clustered_params - baseline_params
    size_delta = clustered_size_mb - baseline_size_mb

    print(f"{'PER':<30} {baseline_results['per']:>11.4f} {clustered_results['per']:>11.4f} {per_delta:>+11.4f}")
    print(f"{'WER':<30} {baseline_results['wer']:>11.4f} {clustered_results['wer']:>11.4f} {wer_delta:>+11.4f}")
    print(f"{'Output Vocab Size':<30} {original_phoneme_count:>12} {cluster_count:>12} {f'-{vocab_reduction:.1f}%':>12}")
    print(f"{'Parameter Count':<30} {baseline_params:>12,} {clustered_params:>12,} {param_delta:>+12,}")
    print(f"{'Checkpoint Size (MB)':<30} {baseline_size_mb:>11.2f} {clustered_size_mb:>11.2f} {size_delta:>+11.2f}")
    print(f"{'Test Samples':<30} {baseline_results['num_samples']:>12} {clustered_results['num_samples']:>12}")

    print("\n" + "=" * 60)
    print("SAMPLE PREDICTIONS (first 5)")
    print("=" * 60)

    for i in range(min(5, len(baseline_results['samples']))):
        b = baseline_results['samples'][i]
        c = clustered_results['samples'][i]
        print(f"\n  Sample {i+1}:")
        print(f"    Baseline  predicted: {b['predicted']}")
        print(f"    Baseline  expected:  {b['expected']}")
        print(f"    Baseline  PER:       {b['per']:.4f}")
        print(f"    Clustered predicted: {c['predicted']}")
        print(f"    Clustered expected:  {c['expected']}")
        print(f"    Clustered PER:       {c['per']:.4f}")

    os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)

    report = f"""# Evaluation Report — Baseline vs. Clustered G2P Model


This report compares the baseline G2P model (trained on raw phonemes) with the
clustered G2P model (trained on cluster labels derived from K-Means phoneme clustering).

Both models use the same Transformer architecture:
- Layers: {NUM_LAYERS}
- d_model: {D_MODEL}
- Attention heads: {NUM_HEADS}
- Feed-forward dim: {DFF}
- Dropout: {DROPOUT_RATE}

Both use the same train/val/test split (80/10/10, random seed=42).


| Metric | Baseline | Clustered | Δ |
|--------|----------|-----------|---|
| PER | {baseline_results['per']:.4f} ({baseline_results['per']*100:.2f}%) | {clustered_results['per']:.4f} ({clustered_results['per']*100:.2f}%) | {per_delta:+.4f} |
| WER | {baseline_results['wer']:.4f} ({baseline_results['wer']*100:.2f}%) | {clustered_results['wer']:.4f} ({clustered_results['wer']*100:.2f}%) | {wer_delta:+.4f} |
| Output Vocab Size | {original_phoneme_count} phonemes | {cluster_count} clusters | −{vocab_reduction:.1f}% |
| Parameter Count | {baseline_params:,} | {clustered_params:,} | {param_delta:+,} |
| Checkpoint Size (MB) | {baseline_size_mb:.2f} | {clustered_size_mb:.2f} | {size_delta:+.2f} |
| Test Samples | {baseline_results['num_samples']} | {clustered_results['num_samples']} | — |


1. **Vocab Reduction:** The phoneme vocabulary was reduced from {original_phoneme_count} to {cluster_count} tokens ({vocab_reduction:.1f}% reduction).

2. **PER Change:** PER {'improved' if per_delta < 0 else 'increased'} by {abs(per_delta)*100:.2f} percentage points.

3. **WER Change:** WER {'improved' if wer_delta < 0 else 'increased'} by {abs(wer_delta)*100:.2f} percentage points.

4. **Model Size:** The clustered model has {'fewer' if param_delta < 0 else 'more'} parameters ({abs(param_delta):,} {'fewer' if param_delta < 0 else 'additional'}) due to the smaller output vocabulary.



| # | Predicted | Expected | PER |
|---|-----------|----------|-----|
"""

    for i, s in enumerate(baseline_results['samples'][:10]):
        report += f"| {i+1} | `{s['predicted']}` | `{s['expected']}` | {s['per']:.4f} |\n"

    report += """

| # | Predicted | Expected | PER |
|---|-----------|----------|-----|
"""

    for i, s in enumerate(clustered_results['samples'][:10]):
        report += f"| {i+1} | `{s['predicted']}` | `{s['expected']}` | {s['per']:.4f} |\n"

    report += """

- PER is computed as `editdistance(predicted, reference) / len(reference)` at the token level.
- WER is binary: 1.0 if the entire predicted sequence differs from reference, 0.0 if exact match.
- Both models were evaluated on the same held-out test set using greedy decoding.
- The clustered model's PER operates at the cluster level — each "error" is a wrong cluster assignment.
"""

    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        f.write(report)
    print(f"\n✅ Saved evaluation report: {REPORT_PATH}")

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Metric", "Baseline", "Clustered", "Delta"])
        writer.writerow(["PER", f"{baseline_results['per']:.4f}", f"{clustered_results['per']:.4f}", f"{per_delta:+.4f}"])
        writer.writerow(["WER", f"{baseline_results['wer']:.4f}", f"{clustered_results['wer']:.4f}", f"{wer_delta:+.4f}"])
        writer.writerow(["Output Vocab Size", original_phoneme_count, cluster_count, f"-{vocab_reduction:.1f}%"])
        writer.writerow(["Parameter Count", baseline_params, clustered_params, param_delta])
        writer.writerow(["Checkpoint Size (MB)", f"{baseline_size_mb:.2f}", f"{clustered_size_mb:.2f}", f"{size_delta:+.2f}"])
        writer.writerow(["Test Samples", baseline_results['num_samples'], clustered_results['num_samples'], "—"])

    print(f"✅ Saved comparison CSV: {CSV_PATH}")
    print(f"\n{'='*60}")
    print("TASK 3 COMPLETE")
    print(f"{'='*60}")


if __name__ == "__main__":
  !pip install editdistance -q
  main()

Mounted at /content/drive
TASK 3: BASELINE vs. CLUSTERED G2P EVALUATION

Loading baseline dataset...
  Loaded 54753 baseline samples
Loading clustered dataset...
  Loaded 54753 clustered samples
  Test set size: 5476 samples

  Baseline  — src vocab: 140, tgt vocab: 61
  Clustered — src vocab: 140, tgt vocab: 43

Building baseline model...
  Loading weights from: /content/drive/MyDrive/best_g2p_transformer.weights.h5
  Parameters: 1,422,141
  Checkpoint: 5.68 MB

Building clustered model...
  Loading weights from: /content/drive/MyDrive/g2p_clustered_model.weights.h5
  Parameters: 1,417,515
  Checkpoint: 5.66 MB

EVALUATING BASELINE MODEL...
  [Baseline] Evaluated 500/5476 samples...
  [Baseline] Evaluated 1000/5476 samples...
  [Baseline] Evaluated 1500/5476 samples...
  [Baseline] Evaluated 2000/5476 samples...
  [Baseline] Evaluated 2500/5476 samples...
  [Baseline] Evaluated 3000/5476 samples...
  [Baseline] Evaluated 3500/5476 samples...
  [Baseline] Evaluated 4000/5476 samples...